# ESG RAG — 02: Asking questions

This notebook adds the **"G" of RAG (Generation)**: an LLM reads the retrieved passages and writes an answer with citations.

```
Question → embedding (bge-m3) → search ChromaDB (top 10 chunks, per company)
         → whole pages for the 5 best pages + the other chunks → LLM → answer + citations
```

Two LLMs are available:

| | 🟢 Qwen3-8B (open source) | 🟣 Claude Haiku 5.5 (API) |
|---|---|---|
| Runs | In Colab, on the GPU (via Ollama) | On Anthropic's servers |
| Cost | Free | ~US$ 0.001–0.002 per question |
| Setup | Downloads ~5 GB every new session | Only an API key |
| Speed | ~10–15 s per answer on a T4 | ~2–3 s per answer |

The retrieval is **exactly the same** for both; only the generation step changes.

**Requirements:** the database built by `01_indexing.ipynb` in `MyDrive/esg-rag/`, and a GPU runtime for Qwen
(`Runtime → Change runtime type → T4`). The project runs in Colab with Google Drive as storage;
`01_indexing.ipynb` explains what to change to run it elsewhere.

## Step 0 — Choose the LLM(s)

One or both models can be turned on.

The **retrieval settings** are the best configuration found in `03_evals.ipynb` ("Run D"): 10 chunks, per-company search
for questions that don't name a company, and whole pages for the 5 most relevant pages plus the other chunks found.

In [1]:
USE_QWEN = True      # open-source model, runs in Colab (free)
USE_CLAUDE = True   # Claude API (needs an API key, see Step 2b)

DEFAULT_LLM = "qwen" if USE_QWEN else "claude"   # used by ask() when you don't choose

# ── Retrieval settings: the best configuration from the Notebook 03 evals (Run D) ──
N_RESULTS = 10                     # chunks retrieved by the search
MULTI_COMPANY_MODE = "per_company" # questions with no company: search each company separately
CONTEXT_MODE = "page_plus_chunks"  # whole pages for the top MAX_PAGES pages + the other chunks ("chunk" = old behavior)
MAX_PAGES = 5

print("Default LLM:", DEFAULT_LLM, "| top-k:", N_RESULTS, "| multi-company:", MULTI_COMPANY_MODE,
      "| context:", CONTEXT_MODE, f"(max {MAX_PAGES} pages)" if CONTEXT_MODE != "chunk" else "")

Default LLM: qwen | top-k: 10 | multi-company: per_company | context: page_plus_chunks (max 5 pages)


## Step 1 — Libraries

`pymupdf4llm` isn't needed here, because the PDFs were already processed in `01_indexing.ipynb`.

In [2]:
%pip install -q anthropic chromadb sentence-transformers requests

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-bigquery 3.47.0 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
holidays 0.108 require

## Step 2a — 🟢 Set up Qwen with Ollama (only if `USE_QWEN = True`)

**Ollama** downloads and runs open-source LLMs. It works like a small local server: the code sends it a question, and it sends back the answer.

This cell:
1. Installs Ollama
2. Starts the Ollama server in the background
3. Downloads `qwen3:8b` (~5 GB, a 4-bit quantized version that fits on the T4)

⏱️ It takes a few minutes and has to run again **every new Colab session**.
On Colab, the installer prints `WARNING: systemd is not running`: this is expected, because the cell starts the server itself.

In [3]:
import subprocess, time, requests

OLLAMA_URL = "http://localhost:11434"
QWEN_MODEL = "qwen3:8b"    # you can try others later, e.g. "qwen3:4b" (faster) or "qwen3:14b" (doesn't fit well on a T4)

def ollama_is_running():
    try:
        requests.get(OLLAMA_URL, timeout=5)
        return True
    except requests.exceptions.RequestException:   # any error: not ready yet
        return False

if USE_QWEN:
    # 1. Install Ollama (zstd is needed by the installer, pciutils lets it detect the GPU)
    !apt-get -qq install -y zstd pciutils > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh

    # 2. Start the server in the background (only if it isn't running yet) and wait until it responds
    if not ollama_is_running():
        ollama_log = open("/content/ollama.log", "w")
        subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=subprocess.STDOUT)
    for _ in range(60):
        if ollama_is_running():
            print("Ollama server is running ✅")
            break
        time.sleep(1)
    else:
        print("⚠️ Ollama didn't start. Run: !tail -20 /content/ollama.log")

    # 3. Download the model
    !ollama pull {QWEN_MODEL}
else:
    print("Skipping Qwen (USE_QWEN = False)")

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Ollama server is running ✅


**Check that Qwen runs on the GPU.** This lists the downloaded models, sends a first message (which loads the model) and shows where it runs.
The `PROCESSOR` column should say **100% GPU**. If it says CPU, the runtime has no GPU: change it (`Runtime → Change runtime type`), restart the session and run Step 2a again.

In [4]:
if USE_QWEN:
    !ollama list
    r = requests.post(f"{OLLAMA_URL}/api/generate",
                      json={"model": QWEN_MODEL, "prompt": "Say hi in Portuguese", "stream": False, "think": False},
                      timeout=600)
    if r.ok:
        print("Qwen says:", r.json()["response"])
    else:
        print("⚠️ Error:", r.status_code, r.text)   # e.g. "model not found" = the download didn't finish
    !ollama ps

NAME        ID              SIZE      MODIFIED      
qwen3:8b    500a1f067a9f    5.2 GB    5 minutes ago    
Qwen says: Olá!
NAME        ID              SIZE      PROCESSOR    CONTEXT    RUNNER      UNTIL              
qwen3:8b    500a1f067a9f    5.6 GB    100% GPU     4096       llamacpp    4 minutes from now    


## Step 2b — 🟣 Set up Claude (only if `USE_CLAUDE = True`)

The API key is read from **Colab Secrets** (the 🔑 icon in the left sidebar), under the name `ANTHROPIC_API_KEY`, with **notebook access** turned on.
Keys are created at https://platform.claude.com (Settings → API Keys).

⚠️ The key is never written in the code, so it can't leak through the notebook or the repository.

In [5]:
CLAUDE_MODEL = "claude-haiku-5-5"   # or "claude-sonnet-5-5" (smarter, ~20x more expensive)

if USE_CLAUDE:
    import anthropic
    from google.colab import userdata
    client_claude = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
    print("Claude client ready ✅")
else:
    print("Skipping Claude (USE_CLAUDE = False)")

Claude client ready ✅


## Step 3 — Load the database from Drive

The reverse of `01_indexing.ipynb`: the database is copied from Drive to Colab's local disk and opened there.
`DB_NAME` chooses which one: `chroma_db_vision` (text + image transcriptions, the best result) or `chroma_db` (text only).

In [6]:
import os
import shutil
import chromadb
from google.colab import drive

drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/esg-rag"
DB_NAME = "chroma_db_vision"   # best database (text + image transcriptions); or "chroma_db" for text only
DRIVE_DB_DIR = f"{PROJECT_DIR}/{DB_NAME}"
LOCAL_DB_DIR = "/content/chroma_db"

# If this cell already ran in this session, ChromaDB still holds the old copy open: forget it first
from chromadb.api.client import SharedSystemClient
SharedSystemClient.clear_system_cache()

shutil.rmtree(LOCAL_DB_DIR, ignore_errors=True)
shutil.copytree(DRIVE_DB_DIR, LOCAL_DB_DIR)

db = chromadb.PersistentClient(path=LOCAL_DB_DIR)
collection = db.get_collection("esg_reports")
print("Chunks in the database:", collection.count())

Mounted at /content/drive
Chunks in the database: 1611


## Step 4 — Load the embedding model

⚠️ The model must be **the same one used in `01_indexing.ipynb`** (`BAAI/bge-m3`).
Questions and chunks have to be turned into numbers the same way, or the distances mean nothing,
like comparing a length in centimeters with one in inches.

In [7]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
embedder = SentenceTransformer("BAAI/bge-m3", device=device)
print("Embedding model loaded on", device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

... (8 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Embedding model loaded on cuda


## Step 5 — Retrieval: find the best passages

This is the "R" of RAG. `retrieve()` returns the chunks most similar to the question, optionally filtered by `company` and/or `year`.

**Multi-company questions** (no `company`): with `MULTI_COMPANY_MODE = "global"`, one company can take most of the slots.
With `"per_company"`, each company is searched separately (10 results / 3 companies → 4 each), so all of them get a fair chance.

**Parent page retrieval** (`CONTEXT_MODE = "page_plus_chunks"`): the search uses small chunks (precise), but the LLM receives
the **whole page** for the 5 most relevant pages, plus the other chunks found (so nothing the search found is dropped).
Whole pages are rebuilt from their chunks in the database, removing the overlaps. `get_context()` runs the whole retrieval step.

In [8]:
def retrieve(question, n_results=5, company=None, year=None):
    # Build the optional filter
    filters = []
    if company:
        filters.append({"company": company})
    if year:
        filters.append({"year": year})
    if len(filters) == 0:
        where = None
    elif len(filters) == 1:
        where = filters[0]
    else:
        where = {"$and": filters}          # ChromaDB syntax for "both conditions"

    query_embedding = embedder.encode([question], normalize_embeddings=True).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        where=where,
    )

    chunks = []
    for text, meta, dist in zip(results["documents"][0],
                                results["metadatas"][0],
                                results["distances"][0]):
        chunks.append({"text": text, "distance": dist, **meta})
    return chunks

# ── Multi-company questions ──────────────────────────────────────────────
# "global": one search over all reports (top-k overall; one company can take most slots)
# "per_company": search each company separately and combine (every company gets a fair share)
COMPANIES = ["natura", "petrobras", "vale"]

def retrieve_per_company(question, n_results=5, year=None, companies=COMPANIES):
    k_per_company = -(-n_results // len(companies))      # rounds up: 5 results / 3 companies -> 2 each
    chunks = []
    for name in companies:
        chunks += retrieve(question, n_results=k_per_company, company=name, year=year)
    return sorted(chunks, key=lambda c: c["distance"])   # most similar first

def smart_retrieve(question, n_results=5, company=None, year=None):
    if company is None and MULTI_COMPANY_MODE == "per_company":
        return retrieve_per_company(question, n_results=n_results, year=year)
    return retrieve(question, n_results=n_results, company=company, year=year)

# ── Parent page retrieval ("page_plus_chunks") ───────────────────────────
# Search with small chunks (precise), then give the LLM the WHOLE page for the most relevant pages,
# plus the other retrieved chunks (so nothing found by the search is dropped).
_page_cache = {}

def get_page_text(company, year, page):
    """Rebuilds a page's text from all its chunks in the database (removing the overlaps)."""
    key = (company, year, page)
    if key not in _page_cache:
        res = collection.get(where={"$and": [{"company": company}, {"year": year}, {"page": page}]})
        # IDs look like "petrobras_2025_p147_c2": sort the chunks by their position on the page
        parts = sorted(zip(res["ids"], res["documents"]), key=lambda x: int(re.search(r"_c(\d+)$", x[0]).group(1)))
        merged = ""
        for _, text in parts:
            # Each chunk starts with the end of the previous one (the overlap): don't repeat it
            pos = merged.rfind(text[:60], max(0, len(merged) - 600)) if merged else -1
            merged = merged[:pos] + text if pos != -1 else (merged + "\n" + text).strip()
        _page_cache[key] = merged
    return _page_cache[key]


def expand_to_pages(chunks, max_pages=5):
    """Unique pages, most relevant first, alternating between companies (so none is left out)."""
    by_company = {}
    for c in chunks:
        pages = by_company.setdefault(c["company"], [])
        if not any(p["page"] == c["page"] and p["year"] == c["year"] for p in pages):
            pages.append(c)
    selected = []
    while len(selected) < max_pages and any(by_company.values()):
        for company in list(by_company):
            if by_company[company] and len(selected) < max_pages:
                selected.append(by_company[company].pop(0))
    return [{**c, "text": get_page_text(c["company"], c["year"], c["page"])} for c in selected]


def get_context(question, n_results=10, company=None, year=None):
    """The full retrieval step: chunks, whole pages, or whole pages + the other chunks (CONTEXT_MODE)."""
    chunks = smart_retrieve(question, n_results=n_results, company=company, year=year)
    if CONTEXT_MODE == "page":
        return expand_to_pages(chunks, max_pages=MAX_PAGES)
    if CONTEXT_MODE == "page_plus_chunks":
        pages = expand_to_pages(chunks, max_pages=MAX_PAGES)
        page_keys = {(p["company"], p["year"], p["page"]) for p in pages}
        leftovers = [c for c in chunks if (c["company"], c["year"], c["page"]) not in page_keys]
        return pages + leftovers
    return chunks

# Quick test
for c in retrieve("emissões de gases de efeito estufa", company="petrobras"):
    print(f"{c['company']} {c['year']}, page {c['page']} (distance {c['distance']:.3f})")

petrobras 2025, page 114 (distance 0.374)
petrobras 2025, page 229 (distance 0.387)
petrobras 2025, page 108 (distance 0.402)
petrobras 2025, page 234 (distance 0.413)
petrobras 2025, page 229 (distance 0.420)


## Step 6 — Build the prompt

The part that matters most for answer quality. The LLM receives:

- **Rules** (the system prompt): use only the sources, cite them, keep the units, and say clearly when the answer isn't in the reports.
- **Numbered sources**: each passage gets a number like `[1]`, with company, year and page, so the LLM can cite it.

The **same prompt** goes to Qwen and Claude, which keeps the comparison fair.

In [9]:
SYSTEM_PROMPT = """You are an assistant that answers questions about companies' sustainability reports.

Rules:
1. Use ONLY the information in the sources provided. Do not use outside knowledge.
2. Cite every fact with the source number in brackets, like [1] or [2][3].
3. If the sources don't contain the answer, say clearly that you couldn't find it in the reports. Never guess.
4. When you give numbers, keep the exact units from the source (e.g., tCO2e, m³, %).
5. Answer in the same language as the question.
6. Be concise: a short paragraph or a few bullet points."""


def build_context(chunks):
    parts = []
    for i, c in enumerate(chunks, start=1):
        header = f"[{i}] {c['company'].capitalize()} — {c['year']} report, page {c['page']}"
        parts.append(f"{header}\n{c['text']}")
    return "\n\n---\n\n".join(parts)


def build_user_message(question, chunks):
    return f"Sources:\n\n{build_context(chunks)}\n\n---\n\nQuestion: {question}"


# See what the LLM will receive
print(build_context(retrieve("water consumption", company="vale", n_results=2))[:1500])

[1] Vale — 2025 report, page 47
###### **Water performance**

In 2025, the Company withdrew 458.3 Mm³ (million cubic meters) of freshwater, representing a 12% increase in total withdrawals compared to 2024. Despite this, we also increased the volume of water withdrawn and returned unused to the environment or allocated to third parties (340.3 Mm³) by 12%, helping reduce pressure on water availability in the watershed.

Recirculated and reused water (611.2 Mm³) accounted for 84% of Vale’s total water demand, reflecting consistent progress in our internal practices. The increase in operational use was associated with production growth, the ramp-up of plants and production lines, and the strengthening of particulate control actions.

As reflected in Vale’s water balance, illustrated below, 340.3 Mm³ (74% of all water withdrawn) is not used in operations. This volume is directed immediately, and without use, to communities and/or to the environment, contributing to local water availability

## Step 7 — Generation: one function for each LLM

Both functions receive the same input and return the same output: the answer, the number of tokens, the cost and the time taken.

**Two details about Qwen:**
- `num_ctx`: Ollama's default context window is small, so it is set explicitly: 8,192 tokens for chunks and 16,384 for whole pages.
  Without it, Ollama silently **cuts off** part of the sources. A warning is printed if the input gets close to the limit.
- `think: False`: Qwen3 can "think" before answering. That is slower and unnecessary here, because the answer is already in the sources.

In [10]:
import re, time, requests   # needed by the functions below (even when Qwen is off)

CLAUDE_PRICES = {   # USD per million tokens (input, output). Check current prices at platform.claude.com
    "claude-haiku-5-5": (0.10, 0.50),
    "claude-sonnet-5-5": (2.00, 10.00),
}


# Whole pages are long, so Qwen needs a bigger window when CONTEXT_MODE isn't "chunk"
QWEN_NUM_CTX = 8192 if CONTEXT_MODE == "chunk" else 16384


def generate_qwen(system_prompt, user_message):
    start = time.time()
    response = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={
            "model": QWEN_MODEL,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_message},
            ],
            "stream": False,
            "think": False,
            "options": {"num_ctx": QWEN_NUM_CTX, "temperature": 0},
        },
        timeout=600,
    )
    response.raise_for_status()
    data = response.json()
    if data.get("prompt_eval_count", 0) >= QWEN_NUM_CTX - 100:
        print(f"⚠️ Qwen's input was probably cut off ({data['prompt_eval_count']} tokens): increase QWEN_NUM_CTX or reduce N_RESULTS")
    answer = data["message"]["content"]
    answer = re.sub(r"<think>.*?</think>", "", answer, flags=re.DOTALL).strip()   # just in case
    return {
        "answer": answer,
        "input_tokens": data.get("prompt_eval_count", 0),
        "output_tokens": data.get("eval_count", 0),
        "cost": 0.0,
        "seconds": time.time() - start,
    }


def generate_claude(system_prompt, user_message):
    start = time.time()
    response = client_claude.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=4096,   # room for the answer (and any internal reasoning, which also counts as output)
        system=system_prompt,
        messages=[{"role": "user", "content": user_message}],
    )
    answer = "".join(block.text for block in response.content if block.type == "text")
    if response.stop_reason == "max_tokens":
        answer += "\n\n⚠️ [Answer cut off: the max_tokens limit was reached]"
    price_in, price_out = CLAUDE_PRICES.get(CLAUDE_MODEL, (0, 0))
    usage = response.usage
    return {
        "answer": answer,
        "input_tokens": usage.input_tokens,
        "output_tokens": usage.output_tokens,
        "cost": (usage.input_tokens * price_in + usage.output_tokens * price_out) / 1_000_000,
        "seconds": time.time() - start,
    }


GENERATORS = {"qwen": generate_qwen, "claude": generate_claude}
ENABLED = {"qwen": USE_QWEN, "claude": USE_CLAUDE}

## Step 8 — `ask()`: the complete RAG

Retrieval, prompt and generation in one call. Qwen runs with `temperature=0`, so its answers are as consistent as possible between runs.

In [11]:
def show_result(name, result):
    print(f"🤖 {name.upper()}  ({result['seconds']:.1f} s · "
          f"{result['input_tokens']} in + {result['output_tokens']} out tokens · "
          f"US$ {result['cost']:.5f})")
    print(result["answer"], "\n")


def show_sources(chunks):
    print("📚 Sources:")
    for i, c in enumerate(chunks, start=1):
        print(f"  [{i}] {c['company'].capitalize()} {c['year']}, page {c['page']}")


def ask(question, llm=None, company=None, year=None, n_results=None):
    llm = llm or DEFAULT_LLM
    if not ENABLED.get(llm):
        raise ValueError(f"'{llm}' is not enabled. Set USE_{llm.upper()} = True in Step 0 and run the setup cells.")

    chunks = get_context(question, n_results=n_results or N_RESULTS, company=company, year=year)   # R
    user_message = build_user_message(question, chunks)                          # A
    result = GENERATORS[llm](SYSTEM_PROMPT, user_message)                       # G

    print("❓", question, "\n")
    show_result(llm, result)
    show_sources(chunks)
    return {"question": question, "llm": llm, "chunks": chunks, **result}

## Step 9 — Examples

Questions in Portuguese and English, with and without a company filter.

In [12]:
result = ask("Quais foram as emissões de gases de efeito estufa da Petrobras em 2025?", company="petrobras")

❓ Quais foram as emissões de gases de efeito estufa da Petrobras em 2025? 

🤖 QWEN  (31.0 s · 9422 in + 36 out tokens · US$ 0.00000)
As emissões de gases de efeito estufa da Petrobras em 2025 foram de 452 milhões de tCO2e [1]. 

📚 Sources:
  [1] Petrobras 2025, page 117
  [2] Petrobras 2025, page 115
  [3] Petrobras 2025, page 34
  [4] Petrobras 2025, page 220
  [5] Petrobras 2025, page 6
  [6] Petrobras 2025, page 116


In [13]:
# A question in English
result = ask("What percentage of Natura's ingredients are renewable?", company="natura")

❓ What percentage of Natura's ingredients are renewable? 

🤖 QWEN  (11.7 s · 7838 in + 30 out tokens · US$ 0.00000)
In 2025, Natura achieved 94.7% renewable or natural-origin ingredients for the Natura brand [1]. 

📚 Sources:
  [1] Natura 2025, page 75
  [2] Natura 2025, page 12
  [3] Natura 2025, page 26
  [4] Natura 2025, page 62
  [5] Natura 2025, page 19
  [6] Natura 2025, page 60
  [7] Natura 2025, page 35
  [8] Natura 2025, page 17


In [14]:
# No filter: the search looks at all companies
result = ask("Which companies mention water consumption targets?")

❓ Which companies mention water consumption targets? 

🤖 QWEN  (18.8 s · 9245 in + 99 out tokens · US$ 0.00000)
Vale and Petrobras mention water consumption targets. Vale aims to reduce specific freshwater use by 27% by 2030 compared to 2017, achieving a 32% reduction in 2025 [1][4]. Petrobras targets a 40% reduction in freshwater capture by 2030 compared to 2021, achieving a 25% reduction in 2025 [2][5]. 

📚 Sources:
  [1] Vale 2025, page 47
  [2] Petrobras 2025, page 147
  [3] Natura 2025, page 82
  [4] Vale 2025, page 18
  [5] Petrobras 2025, page 143
  [6] Natura 2025, page 83
  [7] Natura 2025, page 83


### The "I don't know" test

This question **can't** be answered from sustainability reports.
A reliable RAG says it couldn't find the answer instead of making something up.

In [15]:
result = ask("Qual é o salário do CEO da Vale?", company="vale")

❓ Qual é o salário do CEO da Vale? 

🤖 QWEN  (20.1 s · 10056 in + 150 out tokens · US$ 0.00000)
O relatório da Vale de 2025 não fornece diretamente o salário do CEO. No entanto, na seção sobre remuneração do Executivo, é mencionado que o total de remuneração do Executivo (incluindo compensação fixa, variável e outros benefícios) para o ano fiscal de 2025 foi de BRL 155.487.354 [2]. O CEO é um dos membros do Executivo, e a remuneração total inclui sua compensação. Portanto, o salário do CEO está incluído nesse valor total, mas não é especificado individualmente no relatório. 

📚 Sources:
  [1] Vale 2025, page 63
  [2] Vale 2025, page 96
  [3] Vale 2025, page 35
  [4] Vale 2025, page 10
  [5] Vale 2025, page 31
  [6] Vale 2025, page 2
  [7] Vale 2025, page 33
  [8] Vale 2025, page 92
  [9] Vale 2025, page 107


## Step 10 — ⚔️ Qwen vs. Claude (only if both are enabled)

`compare()` retrieves the passages **once** and sends the **same sources** to both models,
so any difference between the answers comes from the LLM, not from the search.

In [16]:
def compare(question, company=None, year=None, n_results=None):
    active = [name for name, on in ENABLED.items() if on]
    if len(active) < 2:
        print("Turn on both USE_QWEN and USE_CLAUDE in Step 0 to compare.")
        return

    chunks = get_context(question, n_results=n_results or N_RESULTS, company=company, year=year)
    user_message = build_user_message(question, chunks)

    print("❓", question, "\n")
    results = {}
    for name in active:
        results[name] = GENERATORS[name](SYSTEM_PROMPT, user_message)
        show_result(name, results[name])
    show_sources(chunks)
    return {"question": question, "chunks": chunks, "results": results}


comparison = compare("Quais foram as emissões de gases de efeito estufa da Petrobras em 2025?", company="petrobras")

❓ Quais foram as emissões de gases de efeito estufa da Petrobras em 2025? 

🤖 QWEN  (3.0 s · 9422 in + 36 out tokens · US$ 0.00000)
As emissões de gases de efeito estufa da Petrobras em 2025 foram de 452 milhões de tCO2e [1]. 

🤖 CLAUDE  (8.7 s · 12694 in + 1901 out tokens · US$ 0.00222)
Em 2025, a Petrobras reportou as seguintes emissões de gases de efeito estufa:

- **Emissões absolutas operacionais:** 50 milhões de tCO2e, uma redução de 36% em relação a 2015 (78 milhões de tCO2e) [2].
- **Emissões operacionais das atividades de óleo e gás:** 47 milhões de tCO2e, um acréscimo de 3 milhões de tCO2e em relação ao período anterior, atribuído à entrada em operação de novos ativos [2].
- **Intensidade de emissões de GEE no E&P:** 14,7 kgCO2e/boe, 1% abaixo de 2024 [2].
- **Intensidade de emissões de metano no upstream:** 0,23 tCH4/mil tHC [6].
- **Emissões da cadeia de valor (escopo 3):** a Categoria 10 (processamento de produtos vendidos) foi de 22 milhões de tCO2e e a Categoria 11 (uso 

In [17]:
comparison = compare("Qual é o salário do CEO da Vale?", company="vale")

❓ Qual é o salário do CEO da Vale? 

🤖 QWEN  (8.8 s · 10056 in + 150 out tokens · US$ 0.00000)
O relatório da Vale de 2025 não fornece diretamente o salário do CEO. No entanto, na seção sobre remuneração do Executivo, é mencionado que o total de remuneração do Executivo (incluindo compensação fixa, variável e outros benefícios) para o ano fiscal de 2025 foi de BRL 155.487.354 [2]. O CEO é um dos membros do Executivo, e a remuneração total inclui sua compensação. Portanto, o salário do CEO está incluído nesse valor total, mas não é especificado individualmente no relatório. 

🤖 CLAUDE  (3.3 s · 12622 in + 644 out tokens · US$ 0.00158)
Não encontrei o salário individual do CEO da Vale nos relatórios fornecidos. O CEO atual é Gustavo Pimenta [8], mas as fontes não trazem a remuneração dele separadamente.

O que as fontes trazem é o total da remuneração do Comitê Executivo, que inclui o CEO e os demais diretores, em reais brasileiros (sem encargos):

- **2025:** remuneração total de BRL 15

## Step 11 — Custom questions

Any question can be asked here, in Portuguese or English, with or without a `company` filter.
The answers cite the report and page, so they can be checked against the PDF.

In [ ]:
result = ask("Your question here", llm=None, company=None)

## Summary

A complete RAG system with a **swappable LLM**:
- **Retrieval:** bge-m3 + ChromaDB, per-company search, whole pages for the most relevant pages
- **Augmentation:** the passages become numbered sources in the prompt
- **Generation:** Qwen3-8B (open source, local) or Claude Haiku (API), using only those sources, with citations

**Next:** `03_evals.ipynb` runs 114 verified test questions on both models and scores retrieval, answers and citations.

🛠️ **Troubleshooting Qwen:** the Ollama server log is in `/content/ollama.log` (`!tail -20 /content/ollama.log` shows the last lines).